# 1 - Fetching all required files

First we need to fetch all required files from Llama's HuggingFace repository.

For Llama-3.2-1B, we need three files:

- The file containing the model weights themselves
- The file containing the model configuration (number of heads, layers, etc...)
- The file continaing the tokenizer data

In [1]:
required_files_per_model = {
    "Llama-3.2-1B": [
        # the model weights themselves, all layers are packed in this file
        "https://huggingface.co/meta-llama/Llama-3.2-1B/resolve/main/model.safetensors",
        # the model configuration
        "https://huggingface.co/meta-llama/Llama-3.2-1B/resolve/main/config.json",
        # the tokenizer vocabulary
        "https://huggingface.co/meta-llama/Llama-3.2-1B/resolve/main/tokenizer.json"
    ]
}

In [ ]:
import urllib.request
from pathlib import Path

if not Path("HF_TOKEN").exists():
    raise Exception("Request an access token from here: https://huggingface.co/meta-llama/Llama-3.2-1B and put it in an HF_TOKEN file")

for model_name, required_files in required_files_per_model.items():
    model_path = Path("models") / model_name
    model_path.mkdir(exist_ok=True, parents=True)
    for required_file in required_files:
        local_file = model_path / required_file.split("/")[-1]
        if not local_file.exists():
            with open("HF_TOKEN") as f:
                HF_TOKEN = f.read().strip()

            display(f"Downloading {local_file}")

            opener = urllib.request.build_opener()
            opener.addheaders = [("Authorization", f"Bearer {HF_TOKEN}")]
            urllib.request.install_opener(opener)

            urllib.request.urlretrieve(required_file, local_file)

# safetensors conversion

Llama tensors are serialized in `safetensors` format. It is pretty straightforward with 3 main sections:
- A header length
- A JSON header
- All serialized tensors

See [the format description](https://github.com/huggingface/safetensors?tab=readme-ov-file#format) for more details.

For Llama 3 1B model, the tensors are encoded in `bfloat16`.

`bfloat16` cannot be used on CPU, we need to convert them to `float32`. `bfloat16`s are just tuncated `float32`s : the lower bits of the fraction are truncated. We will simply pad each `bfloat16` with two `0` bytes and cast the result to `float32`.

See [the Wikipedia bfloat16 page](https://en.wikipedia.org/wiki/Bfloat16_floating-point_format) for more details

In [41]:
%pip install numpy


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [42]:
def bfloat16_to_float32(bf16buffer: bytes) -> bytearray:
    assert len(bf16buffer) % 2 == 0, "the bfloat16 buffer should have an even number of bytes"
    # bfloat16 are fraction-truncated float32s. just add zeros to convert them to float32
    result = bytearray(len(bf16buffer) * 2)
    for i in range(0, len(bf16buffer), 2):
        # endianness: little-endian
        # BF16: 2 bytes : [ B0, B1 ]
        # Float32: 4bytes : [0, 0, B0, B1]
        result[i * 2 + 2: i * 2 + 4] = bf16buffer[i : i + 2]
    return result

The [wikipedia page](https://en.wikipedia.org/wiki/Bfloat16_floating-point_format#Special_values) tells us that `4049` corresponds approximatively to π (3.140625). `safetensors` file format encode data in little endian byte order, so our example needs to be encoded as `b"\x49\x40"`

In [43]:
import struct

fp32_bytes = bfloat16_to_float32(b"\x49\x40")
display("Should be 3.140625")
struct.unpack("<f", fp32_bytes)

'Should be 3.140625'

(3.140625,)

Now extract all tensors from the `safetensors` file, convert them to float32 and save them in a file on disk

In [ ]:
import mmap
import json

for model_dir in Path("models").iterdir():
    if model_dir.is_dir():
        output_dir = model_dir / "tensors-fp32"
        output_dir.mkdir(parents=True)
        with (model_dir / "model.safetensors").open(mode="rb") as file:
            # See safetensors file format : https://github.com/huggingface/safetensors
            mmaped = mmap.mmap(file.fileno(), 0, prot=mmap.PROT_READ)
            # The 8 first bytes are an int64 representing the header length
            header_size = mmaped[:8]
            header_size = struct.unpack("<Q", header_size)[0] # little-endinan uint64
            # We can now read the header and decode it
            header = mmaped[8 : 8 + header_size]
            header = json.loads(header)

            # Now load each tensor and convert it to float32
            for tensor_name, tensor_metadata in header.items():
                if tensor_name == "__metadata__":
                    continue

                start, end = tensor_metadata["data_offsets"]

                # header size must be taken into account:
                start += 8 + header_size
                end += 8 + header_size

                dtype = tensor_metadata["dtype"]
                print(f"Extracting {tensor_name}, {start}, {end}")
                raw_tensor = mmaped[start: end]
                if dtype == "BF16":
                    raw_tensor = bfloat16_to_float32(raw_tensor)
                with open(output_dir / f"{tensor_name}.raw", mode="wb") as file:
                    file.write(raw_tensor)
            with open(output_dir / "metadata.json", mode="w") as file:
                json.dump(header, file)

Extracting model.embed_tokens.weight, 16808, 525353384
Extracting model.layers.0.input_layernorm.weight, 525353384, 525357480
Extracting model.layers.0.mlp.down_proj.weight, 525357480, 558911912
Extracting model.layers.0.mlp.gate_proj.weight, 558911912, 592466344
Extracting model.layers.0.mlp.up_proj.weight, 592466344, 626020776
Extracting model.layers.0.post_attention_layernorm.weight, 626020776, 626024872
Extracting model.layers.0.self_attn.k_proj.weight, 626024872, 628122024
Extracting model.layers.0.self_attn.o_proj.weight, 628122024, 636510632
Extracting model.layers.0.self_attn.q_proj.weight, 636510632, 644899240
Extracting model.layers.0.self_attn.v_proj.weight, 644899240, 646996392
Extracting model.layers.1.input_layernorm.weight, 646996392, 647000488
Extracting model.layers.1.mlp.down_proj.weight, 647000488, 680554920
Extracting model.layers.1.mlp.gate_proj.weight, 680554920, 714109352
Extracting model.layers.1.mlp.up_proj.weight, 714109352, 747663784
Extracting model.layers.1

Now check that we can load the model and its tensors

In [47]:
import numpy as np

def load_raw_model(path: str):
    model_dir = Path(path)
    model = {}
    with open(model_dir / "metadata.json") as file:
        metadata = json.load(file)
    for tensor_name, tensor_metadata in metadata.items():
        if tensor_name == "__metadata__":
            continue
        file = open(model_dir / f"{tensor_name}.raw", mode="rb")
        mmaped = mmap.mmap(file.fileno(), 0, prot=mmap.PROT_READ)
        model[tensor_name] = np.frombuffer(mmaped, dtype=np.float32).reshape(tensor_metadata["shape"])
    return model

In [48]:
model = load_raw_model("models/Llama-3.2-1B/tensors-fp32")

In [51]:
for name, array in model.items():
    display(f"{name} {array.shape}")

'model.embed_tokens.weight (128256, 2048)'

'model.layers.0.input_layernorm.weight (2048,)'

'model.layers.0.mlp.down_proj.weight (2048, 8192)'

'model.layers.0.mlp.gate_proj.weight (8192, 2048)'

'model.layers.0.mlp.up_proj.weight (8192, 2048)'

...

'model.layers.9.self_attn.k_proj.weight (512, 2048)'

'model.layers.9.self_attn.o_proj.weight (2048, 2048)'

'model.layers.9.self_attn.q_proj.weight (2048, 2048)'

'model.layers.9.self_attn.v_proj.weight (512, 2048)'

'model.norm.weight (2048,)'